In [1]:
import os
os.chdir("..")


In [2]:

import json

path = "cot-planning/results/blocksworld_mystery_5/qwq-32b-steered-full-300-1000-2500-1-l-44-fix-rescale-3/plan_generation_po.json"
path_2 = "cot-planning/results/blocksworld_mystery_5/qwq-32b-steered-full-300-1000-2500-0-fix-rescale-3/plan_generation_po.json"

def load_data(path):
    with open(
        path,
        "r"
    ) as f:
        data = json.load(f)["instances"]
    
    data = {x["instance_id"]: x for x in data}

    return data


data = load_data(path)
data_2 = load_data(path_2)

In [3]:
data["4_10"]

{'instance_id': '4_10',
 'Number of blocks': 4,
 'llm_raw_response': '[PLAN]\nnavigate Block D from Block A\ncalculate Block D\nwhisper Block D\norbit Block D from Block B\nnavigate Block A from Block C\ncalculate Block A\nwhisper Block C\norbit Block C from Block D\nwhisper Block B\norbit Block B from Block A\n[PLAN END]',
 'full_response': ' do an Orbit with B as the first object (since Orbit requires the first object has Legend, and the other has Fountain).\n\nSo, for example, Orbit B from D (since D has Fountain). Let\'s see:\n\nAction 2: Orbit B from D.\n\nPreconditions:\n\n- Fountain on D (yes, since D still has it after Whisper B).\n\n- Legend on B (yes, after Whisper).\n\nSo this is possible.\n\nAfter Orbit B from D:\n\n- The effects are:\n\n   True: Autumn (so Autumn is back on?), Fountain on B (the first object, B now has Fountain?), and B illuminates D (since the first object is B, the other is D).\n\n   False: Fountain on D (so D loses Fountain), Legend on B (so B loses Leg

In [4]:
data_2["4_10"]

{'instance_id': '4_10',
 'Number of blocks': 4,
 'llm_raw_response': '[PLAN]\nwhisper Block B\norbit Block B from Block D\ncalculate Block B\nwhisper Block B\norbit Block B from Block A\nwhisper Block D\norbit Block D from Block C\nwhisper Block A\norbit Block A from Block D\n[PLAN END]',
 'full_response': ' do an Orbit of B from D (since D has Fountain). Let\'s see:\n\nAction 2: Orbit B from D.\n\nPreconditions for Orbit: Fountain on the other object (D has Fountain), and Legend on the object (B has Legend). So yes.\n\nAfter Orbit B from D:\n\n- The effects are:\n\n   - Autumn becomes true (so now Autumn is back on).\n\n   - Fountain on B (the object being orbited, B now has Fountain).\n\n   - B illuminates D (since the orbit is B from D, so B illuminates D? Wait, the Orbit action\'s effect is "Object Illuminates other object". The syntax is Orbit object from another object. So Orbit B from D would mean B is the object, D is the other. So the illumination is B → D.\n\n   - The other (

In [5]:
print(data_2["4_10"]["full_response"])

 do an Orbit of B from D (since D has Fountain). Let's see:

Action 2: Orbit B from D.

Preconditions for Orbit: Fountain on the other object (D has Fountain), and Legend on the object (B has Legend). So yes.

After Orbit B from D:

- The effects are:

   - Autumn becomes true (so now Autumn is back on).

   - Fountain on B (the object being orbited, B now has Fountain).

   - B illuminates D (since the orbit is B from D, so B illuminates D? Wait, the Orbit action's effect is "Object Illuminates other object". The syntax is Orbit object from another object. So Orbit B from D would mean B is the object, D is the other. So the illumination is B → D.

   - The other (D) loses Fountain (so D no longer has Fountain).

   - The object (B) loses Legend (so B no longer has Legend).

So after this:

- B has Fountain (from the effect), and D loses Fountain.

- B now illuminates D (B→D).

- Autumn is now true again (because of the Orbit's effect).

- B's Legend is gone, so it's back to normal (bu

In [6]:
instance_ids = list(data.keys())

c = 0

for instance_id in instance_ids:
    gen = data[instance_id]["full_response"]
    gen_2 = data_2[instance_id]["full_response"]

    if gen != gen_2:
        c += 1

print(c)


295


In [7]:
gen = data["4_1"]["full_response"]
gen_2 = data_2["4_1"]["full_response"]

In [8]:
# Find first difference between gen and gen_2
for i in range(min(len(gen), len(gen_2))):
    if gen[i] != gen_2[i]:
        print(f"First difference at position {i}:")
        print(f"gen:   {gen[i:i+50]}")
        print(f"gen_2: {gen_2[i:i+50]}")
        break
else:
    if len(gen) != len(gen_2):
        print(f"Strings have different lengths: {len(gen)} vs {len(gen_2)}")
        if len(gen) > len(gen_2):
            print(f"gen has extra: {gen[len(gen_2):]}")
        else:
            print(f"gen_2 has extra: {gen_2[len(gen):]}")
    else:
        print("Strings are identical")

First difference at position 15:
gen:   do Orbit A from D. Wait, no, because the effect is
gen_2: Orbit A from D. Wait, no, because the effect is "O
